In [0]:
catalog="Ecommerce"

###***Brands***

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

brz_path=f"{catalog}.bronze.brz_brands"

anomalies={
    "BOOKS":"BKS",
    "GROCERY":"GRCY",
    "TOYS":"TOY",
}


df_slv = spark.table(brz_path)

df_slv=df_slv.withColumn('brand_name',trim(col("brand_name")))
df_slv = df_slv.withColumn("brand_code", regexp_replace(col("brand_code"), "[^a-zA-Z0-9]", ""))
df_slv = df_slv.replace(to_replace=anomalies, subset=["category_code"])


df_slv.write.format('delta').mode('overwrite').option('mergeSchema','true').saveAsTable(f'{catalog}.silver.slv_brands')

###***Category***

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *

brz_path=f"{catalog}.bronze.brz_category"
window_fun=Window.partitionBy("category_code").orderBy(col("Ingested_at").asc())


df_slv=spark.table(brz_path)
df_slv=df_slv.withColumn('category_code',upper(col("category_code")))

df_slv=df_slv.withColumn('row_number',row_number().over(window_fun))
df_slv=df_slv.filter(col("row_number")==1)
df_slv=df_slv.drop(col('row_number'))


df_slv.write.format('delta').mode('overwrite').option('mergeSchema',True)\
    .saveAsTable(f'{catalog}.silver.slv_Category')



###***Customers***

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *

brz_path=f"{catalog}.bronze.brz_customers"

df_slv=spark.table(brz_path)

df_slv=df_slv.dropna(subset=['customer_id'])
df_slv=df_slv.fillna('Not Available',subset=['phone'])
df_slv.display()

df_slv.write.format('delta').mode('overwrite').option('mergeSchema',True)\
    .saveAsTable(f'{catalog}.silver.slv_customers')


###***Products***

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *

brz_path=f"{catalog}.bronze.brz_products"

df_slv=spark.table(brz_path)
df_slv=df_slv.withColumn('weight_grams',regexp_replace(col('weight_gr'),'g','').cast(IntegerType()))\
.withColumn('length_cm',regexp_replace(col('length_cm'),',','.').cast(FloatType()))
df_slv=df_slv.withColumn('category_code',upper(col("category_code")))\
    .withColumn('brand_code',upper(col('brand_code')))
df_slv=df_slv.withColumn(
    "material",
      when(col("material") == "Coton", "Cotton")
     .when(col("material") == "Alumium", "Aluminum")
     .when(col("material") == "Ruber", "Rubber")
     .otherwise(col("material"))
)
df_slv=df_slv.withColumn('rating_count',when(col('rating_count').isNotNull(),abs(col('rating_count')))\
    .otherwise(lit(0)))
df_slv=df_slv.drop('weight_gr')
df_slv=df_slv.select('product_id','sku','category_code','brand_code','color','size','material','length_cm','width_cm','height_cm','weight_grams','rating_count','Source_file','Ingested_at')


df_slv.write.format('delta').mode('overwrite').option('mergeSchema',True).saveAsTable(f'{catalog}.silver.slv_products')

###***Dates***

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *

brz_path=f"{catalog}.bronze.brz_dates"
window_fun=Window.partitionBy(col('date')).orderBy(col('date'))

df_slv=spark.read.table(brz_path)
df_slv=df_slv.withColumn('row_num',row_number().over(window_fun)).filter(col('row_num')==1).drop(col('row_num'))
df_slv=df_slv.withColumn('day_name',initcap(col('day_name')))\
    .withColumn('week_of_year',abs(col('week_of_year')))\
    .withColumn('quarter',concat(lit('Q'),col('quarter'),lit('-'),col('year')))\
    .withColumn('Week_of_year',concat(lit('Week'),col('week_of_year'),lit('-'),col('year')))
df_slv=df_slv.withColumnRenamed('Week_of_year','Week')

df_slv.write.format('delta').mode('overwrite').option('mergeSchema',True).saveAsTable(f'{catalog}.silver.slv_dates')